# Omnibus — what the demand model predicts (day-ahead surface)

Reads a `data/demand/demand_<date>.json` produced by `pipeline/predict_demand.py`: every
known stop × 96 ticks (15-min), split into **`baseline_s`** (GBM, event-free normal day) and
**`event_s`** (profile-driven event kernel), summed into **`pressure_s`**.

Two views:
1. **System pulse** — city-wide pressure over the day, baseline vs event stacked.
2. **Per-minute event curves** — each event leg at 1-min resolution from the companion
   `_events.json`, showing the raw smooth pulse shapes.

## 1 — System pulse: baseline vs event

Sum every stop's series, tick by tick. The blue block is the model's idea of a normal day;
the orange block on top is everything the event kernel adds — morning inbound to the venue,
evening outbound after the final whistle.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATE = "2025-07-28"  # day with the Jahn matchday demo event
surf = json.loads(Path(f"../data/demand/demand_{DATE}.json").read_text())

ticks = surf["tick_times"]              # ['00:00', '00:15', ...] len 96
N = surf["n_ticks"]
nodes = surf["nodes"]
xt = np.arange(N)

event_legs = [(n["stop_code"], e["event_label"], e["leg"]) for n in nodes for e in n["events"]]
event_stops = {n["stop_code"] for n in nodes if n["events"]}
print(f"{DATE}: {len(nodes)} stops × {N} ticks")
print(f"event legs: {event_legs}")

In [ ]:
baseline_tot = np.sum([n["baseline_s"] for n in nodes], axis=0)
event_tot = np.sum([n["event_s"] for n in nodes], axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
ax.stackplot(xt, baseline_tot, event_tot,
             labels=["baseline (GBM, normal day)", "event_s (kernel)"],
             colors=["#4575b4", "#d73027"], alpha=0.9)
ax.set_xlim(0, N - 1)
ticks_idx = range(0, N, 8)  # every 2h
ax.set_xticks(list(ticks_idx)); ax.set_xticklabels([ticks[i] for i in ticks_idx])
ax.set_xlabel("time of day"); ax.set_ylabel("city-wide predicted pressure [s]")
ax.set_title(f"Predicted demand pulse — {DATE}  (baseline + event)")
ax.legend(loc="upper left"); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

## 2 — Per-minute event curves (from `demand_<date>_events.json`)

The event kernel is analytic, so the companion `_events.json` exports each leg at
**1-minute** resolution. These are the raw smooth double-logistic pulses — inbound builds
gradually and cuts off at kickoff, outbound spikes at the whistle and clears slowly.
Dashed lines mark kickoff and final whistle.

In [ ]:
ev_doc = json.loads(Path(f"../data/demand/demand_{DATE}_events.json").read_text())
res = ev_doc["resolution_min"]


def to_min(s):  # "HH:MM" -> minute of day
    h, m = s.split(":"); return int(h) * 60 + int(m)


for e in ev_doc["events"]:
    fig, ax = plt.subplots(figsize=(13, 5))
    for lg in e["legs"]:
        y = lg["pressure_s"]
        x0 = to_min(lg["start"])
        x = [x0 + i * res for i in range(len(y))]
        inbound = lg["to"] == e["venue"]
        c = "#d73027" if inbound else "#1a9850"
        lab = f"{lg['from']}→{lg['to']}  ({'inbound' if inbound else 'outbound'}, peak {lg['peak_s']:.0f}s)"
        ax.fill_between(x, y, color=c, alpha=0.2)
        ax.plot(x, y, color=c, lw=2, label=lab)

    ks, ws = to_min(e["event_start"]), to_min(e["event_end"])
    ymax = ax.get_ylim()[1]
    for xm, txt in [(ks, "kickoff"), (ws, "whistle")]:
        ax.axvline(xm, ls="--", c="0.3", alpha=0.7)
        ax.text(xm + 2, ymax * 0.96, txt, fontsize=8, va="top")

    lo = min(to_min(lg["start"]) for lg in e["legs"])
    hi = max(to_min(lg["start"]) + len(lg["pressure_s"]) * res for lg in e["legs"])
    tk = range((lo // 30) * 30, hi + 30, 30)
    ax.set_xticks(list(tk)); ax.set_xticklabels([f"{t // 60:02d}:{t % 60:02d}" for t in tk], rotation=45, fontsize=8)
    ax.set_xlim(lo - 10, hi + 10)
    ax.set_xlabel("time of day"); ax.set_ylabel(f"event pressure [s] ({res}-min)")
    ax.set_title(f"{e['label']} — per-minute event curves")
    ax.legend(loc="upper left"); ax.grid(alpha=0.3)
    fig.tight_layout(); plt.show()